# 04 — Exploratory Data Analysis

## Imports
Load required libraries for data manipulation and visualization.

In [5]:
# If you get a TypeError in Figure 6, try upgrading matplotlib by following
!pip install --upgrade matplotlib

from __future__ import annotations
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.patches import Patch

from nba_project_utils import (
    FIGURES_DIR,
    JOIN_REPORT_FILE,
    JOINED_FILE,
    ensure_figures_dir,
    season_sort_key,
)

zsh:1: command not found: pip


## Settings
Set plot style, define the list of features for correlation analysis, and set salary tier order and colors.

In [6]:
plt.style.use("ggplot")

FEATURES_FOR_CORRELATION = [
    "salary_cap_pct",
    "age",
    "gp",
    "min",
    "w_pct",
    "off_rating",
    "def_rating",
    "net_rating",
    "ast_pct",
    "reb_pct",
    "ts_pct",
    "usg_pct",
    "pie",
    "fgm_pg",
    "fg_pct",
]

TIER_ORDER = ["bench", "starter", "star", "max"]
TIER_COLORS = {
    "bench": "#6B7280",
    "starter": "#2563EB",
    "star": "#059669",
    "max": "#DC2626",
}


## Helper Function
Save the current figure to the figures directory and close the plot.

In [7]:
def save_current_figure(filename: str) -> None:
    """Save the current matplotlib figure to the specified filename within the figures directory, ensuring tight layout and high resolution."""
    path = FIGURES_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()
    print(f"Saved {path}")

## Load Data
Read the joined CSV and filter to the analysis sample only.

In [8]:
ensure_figures_dir()
joined = pd.read_csv(JOINED_FILE)
analysis = joined[joined["analysis_sample"].astype(bool)].copy()
print(f"Analysis sample: {len(analysis):,} rows")

Analysis sample: 2,973 rows


## Figure 1: Salary Distribution
Show how player salaries are spread as a share of the salary cap. Most players sit in the lower range, with a long right tail toward max contracts.

In [10]:
def plot_salary_distribution(data: pd.DataFrame) -> None:
    """ Create and save a histogram of player salary as a percentage of the salary cap, with a vertical line indicating the median value."""
    plt.figure(figsize=(7, 2))
    plt.hist(data["salary_cap_pct"], bins=35, color='#2563EB', edgecolor='white')
    plt.axvline(data["salary_cap_pct"].median(), color='#111827', linewidth=2, label='Median')
    plt.xlim(0, 35)
    plt.title('Distribution of Player Salary as Percent of Cap', fontsize=10)
    plt.xlabel('Salary cap percentage', fontsize=8)
    plt.ylabel('Count', fontsize=8)
    plt.tick_params(labelsize=8)
    plt.legend(fontsize=8)
    save_current_figure('01_salary_cap_pct_distribution.png')

plot_salary_distribution(analysis)

Saved /Users/ryanmccurry/Desktop/nba_modeling_project/src/data/figures/01_salary_cap_pct_distribution.png


## Figure 2: Salary Tiers by Season
Show the count of players in each salary tier (bench / starter / star / max) for each season from 2016-17 onward.

In [13]:
def plot_salary_tiers_by_season(data: pd.DataFrame) -> None:
    """Create and save a stacked bar chart showing the percentage composition of salary tiers for each season in the dataset."""
    tier_counts = (
        data.groupby(["season", "salary_tier"], observed=False)
        .size()
        .unstack(fill_value=0)
        .reindex(columns=TIER_ORDER, fill_value=0)
        .sort_index(key=lambda idx: idx.map(season_sort_key))
    )
    tier_pct = tier_counts.div(tier_counts.sum(axis=1), axis=0) * 100
    ax = tier_pct.plot(
        kind="bar",
        stacked=True,
        figsize=(7, 2),
        color=[TIER_COLORS[tier] for tier in TIER_ORDER],
        width=0.78,
        fontsize=8,
    )
    ax.set_title("Salary Tier Composition by Season", fontsize=10)
    ax.set_xlabel("Season", fontsize=8)
    ax.set_ylabel("% of players", fontsize=8)
    ax.set_ylim(0, 100)
    ax.set_yticks([0, 25, 50, 75, 100])
    ax.set_yticklabels(["0%", "25%", "50%", "75%", "100%"], fontsize=8)
    ax.legend(title="Salary tier", bbox_to_anchor=(1.01, 1), loc="upper left", borderaxespad=0, fontsize=8, title_fontsize=8)
    ax.set_xticklabels([f"'{label.get_text()[-2:]}" for label in ax.get_xticklabels()], rotation=0)
    save_current_figure("02_salary_tier_comp_by_season.png")

plot_salary_tiers_by_season(analysis)

Saved /Users/ryanmccurry/Desktop/nba_modeling_project/src/data/figures/02_salary_tier_comp_by_season.png


## Figure 3: Correlation Heatmap
Show pairwise correlations between salary cap percentage and all player performance metrics to spot which stats are most tied to pay.

In [14]:
def plot_correlation_heatmap(data: pd.DataFrame) -> None:
    """Create and save a heatmap showing the correlation between salary cap percentage and various player performance metrics."""
    features = [feature for feature in FEATURES_FOR_CORRELATION if feature in data.columns]
    corr = data[features].corr(numeric_only=True)

    plt.figure(figsize=(10, 8))
    image = plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
    plt.colorbar(image, fraction=0.046, pad=0.04, label="Correlation")
    plt.xticks(np.arange(len(features)), features, rotation=45, ha="right")
    plt.yticks(np.arange(len(features)), features)
    plt.title("Correlation Heatmap for Salary and Player Metrics")
    save_current_figure("03_correlation_heatmap.png")

plot_correlation_heatmap(analysis)

Saved /Users/ryanmccurry/Desktop/nba_modeling_project/src/data/figures/03_correlation_heatmap.png


## Figure 4: PIE vs Salary Cap Percentage (EDA only, not used in report)
Define a reusable scatter plot function colored by salary tier, then plot PIE (Player Impact Estimate) against salary cap percentage. This was exploratory and not included in the final report.

In [15]:
def plot_scatter_by_tier(data: pd.DataFrame, x_col: str, filename: str, title: str) -> None:
    """Create and save a scatter plot comparing a specified player metric against salary cap percentage, with points colored by salary tier."""
    plt.figure(figsize=(8.5, 5.5))
    for tier in TIER_ORDER:
        tier_data = data[data["salary_tier"].eq(tier)]
        if tier_data.empty:
            continue
        plt.scatter(
            tier_data[x_col],
            tier_data["salary_cap_pct"],
            s=28,
            alpha=0.55,
            color=TIER_COLORS[tier],
            label=tier,
            edgecolors="none",
        )
    plt.title(title)
    plt.xlabel(x_col)
    plt.ylabel("Salary cap percentage")
    plt.legend(title="Salary tier")
    save_current_figure(filename)

plot_scatter_by_tier(
    analysis,
    "pie",
    "04_pie_vs_salary_cap_pct.png",
    "PIE vs Salary Cap Percentage",
)

Saved /Users/ryanmccurry/Desktop/nba_modeling_project/src/data/figures/04_pie_vs_salary_cap_pct.png


## Figure 5: Field Goals Made per Game vs Salary Cap Percentage (EDA only, not used in report)
Plot field goals made per game against salary cap percentage by tier. Higher scoring output tends to come with higher pay, but the spread within tiers is wide. This was exploratory and not included in the final report.

In [16]:
plot_scatter_by_tier(
    analysis,
    "fgm_pg",
    "05_fgm_pg_vs_salary_cap_pct.png",
    "Field Goals Made per Game vs Salary Cap Percentage",
)

Saved /Users/ryanmccurry/Desktop/nba_modeling_project/src/data/figures/05_fgm_pg_vs_salary_cap_pct.png


## Figure 6: Salary Distribution by Tier (EDA only, not used in report)
Compare the spread of salary cap percentages across all four tiers using box plots. Outliers are hidden to keep the focus on the core distribution. This was exploratory and not included in the final report.

In [17]:
def plot_salary_boxplot_by_tier(data: pd.DataFrame) -> None:
    """ Create and save a boxplot comparing the distribution of salary cap percentage across different salary tiers, with boxes colored by tier and outliers hidden."""
    values = [
        data.loc[data["salary_tier"].eq(tier), "salary_cap_pct"].dropna().to_numpy()
        for tier in TIER_ORDER
    ]
    plt.figure(figsize=(8, 5.5))
    box = plt.boxplot(values, tick_labels=TIER_ORDER, patch_artist=True, showfliers=False)
    for patch, tier in zip(box["boxes"], TIER_ORDER):
        patch.set_facecolor(TIER_COLORS[tier])
        patch.set_alpha(0.75)
    plt.title("Salary Cap Percentage by Salary Tier")
    plt.xlabel("Salary tier")
    plt.ylabel("Salary cap percentage")
    save_current_figure("06_salary_cap_pct_by_tier_boxplot.png")

plot_salary_boxplot_by_tier(analysis)

Saved /Users/ryanmccurry/Desktop/nba_modeling_project/src/data/figures/06_salary_cap_pct_by_tier_boxplot.png


## Continued in 05_simple_regression.ipynb
The next three figures (regression fit, coefficient importance, and residuals) are generated in [`05_simple_regression.ipynb`](05_simple_regression.ipynb), where the salary prediction model is built and evaluated.

## Figures 10–12: Age vs Performance and Pay (Figure 10 used in report)
Scatter plots of `age` against `PIE`, `USG_PCT`, and `salary_cap_pct`, with points colored by salary tier, to explore how player impact, usage, and pay vary across the aging curve. Only the age vs PIE plot (Figure 10) was used in the final report; the other two were exploratory.

In [19]:
def plot_scatter_by_tier(
    data: pd.DataFrame,
    x_col: str,
    y_col: str,
    x_label: str,
    y_label: str,
    title: str,
    filename: str,
) -> None:
    """Create and save a scatter plot comparing a specified player metric against salary cap percentage, with points colored by salary tier."""

    plt.figure(figsize=(10, 5.5))
    for tier in TIER_ORDER:
        tier_data = data[data["salary_tier"].eq(tier)]
        if tier_data.empty:
            continue
        plt.scatter(
            tier_data[x_col],
            tier_data[y_col],
            s=28,
            alpha=0.55,
            color=TIER_COLORS[tier],
            label=tier,
            edgecolors="none",
        )
    plt.title(title)
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.legend(title="Salary tier")
    plt.tight_layout()
    save_current_figure(filename)


In [20]:
# Plot 3 graphs ages AGE vs [PIE, USG_PCT, SALARY_CAP_PCT]

plots = [
    ("pie",            "Player Impact Estimate (PIE)", "Age vs PIE",                   "10_age_vs_pie.png"),
    ("usg_pct",        "Usage Percentage",             "Age vs Usage Percentage",      "11_age_vs_usg_pct.png"),
    ("salary_cap_pct", "Salary cap percentage",        "Age vs Salary Cap Pct",        "12_age_vs_salary_cap_pct.png"),
]

for y_col, y_label, title, filename in plots:
    plot_scatter_by_tier(analysis, "age", y_col, "Age", y_label, title, filename)

Saved /Users/ryanmccurry/Desktop/nba_modeling_project/src/data/figures/10_age_vs_pie.png
Saved /Users/ryanmccurry/Desktop/nba_modeling_project/src/data/figures/11_age_vs_usg_pct.png
Saved /Users/ryanmccurry/Desktop/nba_modeling_project/src/data/figures/12_age_vs_salary_cap_pct.png


## Figure 13: Production vs. Pay by Age Group (Indexed to League Average)
Grouped bar chart comparing each age group's average PIE and `salary_cap_pct` against the league-wide average (index = 1.0). Highlights that players aged 19-22 are paid well below their production level, while players aged 30+ are paid above theirs.

In [22]:
def plot_age_group_index(group_index: pd.DataFrame, filename: str) -> None:
    x = np.arange(len(group_index.index))
    width = 0.35

    plt.figure(figsize=(10,4))
    plt.bar(x - width/2, group_index["pie"], width, label="PIE index", color="#DC2626", alpha=0.85)
    plt.bar(x + width/2, group_index["salary_cap_pct"], width, label="Salary cap % index", color="#2563EB", alpha=0.85)

    plt.axhline(1.0, color="gray", linestyle="--", linewidth=1, alpha=0.7)
    plt.xticks(x, group_index.index)
    plt.ylabel("Index (league average = 1.0)")
    plt.xlabel("Age group")
    plt.title("Production vs. Pay by Age Group (Indexed to League Average)")
    plt.legend()
    plt.tight_layout()
    save_current_figure(filename)

age_bins = [18, 22, 25, 29, 33, 100]
age_labels = ["19-22", "23-25", "26-29", "30-33", "34+"]
analysis["age_group"] = pd.cut(analysis["age"], bins=age_bins, labels=age_labels)

group_means = analysis.groupby("age_group", observed=True)[["pie", "salary_cap_pct"]].mean()
overall_means = analysis[["pie", "salary_cap_pct"]].mean()
group_index = group_means / overall_means

plot_age_group_index(group_index, "13_age_group_pie_vs_salary_index.png")

Saved /Users/ryanmccurry/Desktop/nba_modeling_project/src/data/figures/13_age_group_pie_vs_salary_index.png


## Figure 14: Contract Year Comparison (EDA only, not used in report)
Compare mean PIE, true shooting percentage, and field goals made per game between contract-year and non-contract-year players. Error bars show 95% confidence intervals. This was exploratory and not included in the final report.

In [23]:
def plot_contract_year_comparison(data: pd.DataFrame) -> None:
    """Create and save a bar chart comparing the mean values of key performance metrics between contract years and non-contract years, with error bars representing 95% confidence intervals."""
    metrics = ["pie", "ts_pct", "fgm_pg", "usg_pct"]
    labels = ["Non-contract year", "Contract year"]
    x = np.arange(len(metrics))
    width = 0.34

    means = {}
    errors = {}
    for flag in [0, 1]:
        subset = data[data["is_contract_year"].eq(flag)]
        means[flag] = [subset[metric].mean() for metric in metrics]
        errors[flag] = [
            1.96 * subset[metric].std(ddof=1) / np.sqrt(subset[metric].notna().sum())
            for metric in metrics
        ]

    plt.figure(figsize=(8.5, 5.5))
    plt.bar(
        x - width / 2,
        means[0],
        width,
        yerr=errors[0],
        label=labels[0],
        color="#6B7280",
        capsize=4,
    )
    plt.bar(
        x + width / 2,
        means[1],
        width,
        yerr=errors[1],
        label=labels[1],
        color="#2563EB",
        capsize=4,
    )
    plt.title("Exploratory Contract-Year Comparison")
    plt.ylabel("Mean metric value")
    plt.xticks(x, metrics)
    plt.legend()
    save_current_figure("14_contract_year_metric_comparison.png")

plot_contract_year_comparison(analysis)

Saved /Users/ryanmccurry/Desktop/nba_modeling_project/src/data/figures/14_contract_year_metric_comparison.png


## Contract Year T-Test Analysis
Filters to players with at least one contract year and splits their seasons into contract year vs. non-contract year groups. For each metric (PIE, true shooting percentage, field goals made per game, and usage rate), runs an independent t-test and computes the percent difference between contract and non-contract year means. Results are stored in `ttest_table`, which is used to generate Figure 15.

In [24]:
# Choose metrics (excluding mins) to top-correlate with salary
metrics = ["pie", "ts_pct", "fgm_pg", "usg_pct"]

# Filter to players who have at least one contract year
cy_players = analysis[analysis["is_contract_year"]==1]['player'].unique()
target = analysis[analysis['player'].isin(cy_players)]

# Split into contract year and non-contract year
contract = target[target["is_contract_year"]==1]
non_contract = target[target["is_contract_year"]==0]

result = []

for metric in metrics:
    # Conduct t-test
    t_stat, p_val = stats.ttest_ind(contract[metric], non_contract[metric])

    result.append({
        "metric":            metric,
        "contract_mean":     contract[metric].mean(),
        "non_contract_mean": non_contract[metric].mean(),
        "t_stat":            t_stat,
        "p_value":           p_val,
        "reject_or_not":     "reject" if p_val < 0.05 else "not rejected"
    })

ttest_table = pd.DataFrame(result).set_index("metric").round(4)

# Calculate % difference between contract and non-contract years
ttest_table["pct_diff"] = (
    (ttest_table["contract_mean"] - ttest_table["non_contract_mean"])
    / ttest_table["non_contract_mean"] * 100
).round(2)

ttest_table

,contract_mean,non_contract_mean,t_stat,p_value,reject_or_not,pct_diff
metric,,,,,,
pie,0.1025,0.1019,0.3811,0.7031,not rejected,0.59
ts_pct,0.5776,0.5708,2.8838,0.0040,reject,1.19
fgm_pg,4.5697,4.6697,-0.9176,0.3589,not rejected,-2.14
usg_pct,0.1925,0.1956,-1.1420,0.2536,not rejected,-1.58


## Export T-Test Results
Exports `ttest_table` (used in Figure 15) to `ttest_table.csv` for reference and reproducibility.

In [25]:
# Export to CSV
if ttest_table.empty:
    print("No data to export.")
else:
    output_file = "./data/ttest_table.csv"
    ttest_table.to_csv(output_file, encoding="utf-8")

    print("CSV export complete:", output_file)
    print("Rows:", len(ttest_table))
    print("Columns:", len(ttest_table.columns))

CSV export complete: ./data/ttest_table.csv
Rows: 4
Columns: 6


## Figure 15: Contract Year Effect on Performance Metrics
Bar chart showing the percent change in PIE, true shooting percentage, field goals made per game, and usage rate between contract and non-contract years, with bars colored to indicate direction of change and statistical significance.

In [26]:
def plot_contract_year_pct_diff(ttest_table: pd.DataFrame, filename: str) -> None:
    """Create and save a bar chart showing the percentage difference in key performance metrics between contract years and non-contract years, with bars colored to indicate statistical significance and direction of change."""
    plt.figure(figsize=(10, 5.5))
    
    colors = []
    for _, row in ttest_table.iterrows():
        if row["reject_or_not"] == "reject":
            colors.append("#16A34A")
        elif row["pct_diff"] >= 0:
            colors.append("#2563EB")
        else:
            colors.append("#DC2626")

    plt.bar(ttest_table.index, ttest_table["pct_diff"], color=colors, alpha=0.85)
    plt.axhline(0, color="gray", linewidth=1)
    plt.ylabel("% Change (Contract Year vs. Non-Contract Year)")
    plt.xlabel("Metric")
    plt.title("Contract Year Effect on Performance Metrics")

    legend_handles = [
        Patch(facecolor="#2563EB", alpha=0.85, label="Positive change"),
        Patch(facecolor="#DC2626", alpha=0.85, label="Negative change"),
        Patch(facecolor="#16A34A", alpha=0.85, label="Significant (p < 0.05)"),
    ]
    plt.legend(handles=legend_handles)
    
    plt.tight_layout()
    save_current_figure(filename)

plot_contract_year_pct_diff(ttest_table, "15_contract_year_pct_diff.png")

Saved /Users/ryanmccurry/Desktop/nba_modeling_project/src/data/figures/15_contract_year_pct_diff.png
